In [ ]:
# ============================================================================
# 脑MRI体素分类 - TabNet迁移学习
# 严格要求：
# 1.  PyTorch Lightning实现逐步解冻
# 2.  使用预训练文件 (network.pt + model_params.json)
# 3.  监控train/val/test的Loss和Macro F1
# 4.  数据标准化
# ============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback, ModelCheckpoint, EarlyStopping, LearningRateMonitor
from torch.utils.data import DataLoader, TensorDataset
import h5py
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import json
import os
from collections import OrderedDict
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pandas as pd
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
pl.seed_everything(42)

# ============================================================================
# Part 1: 数据模块 - 包含标准化
# ============================================================================

class BrainVoxelDataModule(pl.LightningDataModule):
    """
    数据模块 - 严格实现数据加载和标准化
     标准化：仅在训练集上fit，应用到所有数据集
    """
    
    def __init__(self, data_path: str, batch_size: int = 256, num_workers: int = 4):
        super().__init__()
        self.data_path = data_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.scaler = StandardScaler()  #  标准化器
        
    def setup(self, stage: Optional[str] = None):
        """加载数据并进行标准化"""
        print("="*60)
        print("📂 数据加载与预处理")
        print("="*60)
        
        # 加载原始数据
        with h5py.File(self.data_path, 'r') as f:
            train_data = np.array(f['data']).transpose()
            train_region = np.array(f['region']).transpose() 
            prob_idx = np.array(f['prob_idx']).transpose()
            
        print(f" 原始数据形状: {train_data.shape}")
        print(f" 原始标签形状: {train_region.shape}")
        
        # 分割数据集
        test_indices = np.where(prob_idx == 38)[0]
        train_val_indices = np.where(prob_idx != 38)[0]
        
        # 测试集
        X_test = train_data[test_indices, :]
        y_test = train_region[test_indices, :]
        
        # 训练+验证集
        train_val_data = train_data[train_val_indices, :]
        train_val_labels = train_region[train_val_indices, :]
        
        # 分层划分训练集和验证集
        X_train, X_val, y_train, y_val = train_test_split(
            train_val_data, train_val_labels,
            test_size=0.2,
            random_state=42,
            stratify=np.argmax(train_val_labels, axis=1)
        )
        
        print(f"\n 数据集划分:")
        print(f"   训练集: {X_train.shape[0]} 样本")
        print(f"   验证集: {X_val.shape[0]} 样本")
        print(f"   测试集: {X_test.shape[0]} 样本")
        
        #  标准化 - 关键步骤
        print(f"\n🔧 应用标准化...")
        print(f"   1. 在训练集上fit StandardScaler")
        self.scaler.fit(X_train)  # 仅在训练集上fit
        
        print(f"   2. Transform所有数据集")
        X_train_scaled = self.scaler.transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        X_test_scaled = self.scaler.transform(X_test)
        
        # 验证标准化效果
        print(f"\n 标准化验证:")
        print(f"   训练集均值: {np.mean(X_train_scaled):.6f} (应接近0)")
        print(f"   训练集标准差: {np.std(X_train_scaled):.6f} (应接近1)")
        print(f"   验证集均值: {np.mean(X_val_scaled):.6f}")
        print(f"   测试集均值: {np.mean(X_test_scaled):.6f}")
        
        # 创建PyTorch数据集
        self.train_dataset = TensorDataset(
            torch.FloatTensor(X_train_scaled),
            torch.FloatTensor(y_train)
        )
        self.val_dataset = TensorDataset(
            torch.FloatTensor(X_val_scaled),
            torch.FloatTensor(y_val)
        )
        self.test_dataset = TensorDataset(
            torch.FloatTensor(X_test_scaled),
            torch.FloatTensor(y_test)
        )
        
        print(f"\n 数据预处理完成！")
        
    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False
        )
    
    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False
        )
    
    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
            persistent_workers=True if self.num_workers > 0 else False
        )

# ============================================================================
# Part 2: 预训练模型加载器
# ============================================================================

class PretrainedTabNetLoader:
    """
     加载预训练TabNet模型
    - 读取 network.pt (权重)
    - 读取 model_params.json (架构参数)
    """
    
    def __init__(self, pretrained_dir: str = 'tabnet_model_test0'):
        self.pretrained_dir = pretrained_dir
        self.network_path = os.path.join(pretrained_dir, 'network.pt')
        self.params_path = os.path.join(pretrained_dir, 'model_params.json')
        
        # 验证文件存在
        assert os.path.exists(self.network_path), f"找不到 {self.network_path}"
        assert os.path.exists(self.params_path), f"找不到 {self.params_path}"
        
        print("="*60)
        print("📦 加载预训练模型")
        print("="*60)
        
    def load(self) -> Tuple[Dict, Dict]:
        """加载模型参数和权重"""
        # 1. 加载模型架构参数
        with open(self.params_path, 'r') as f:
            model_params = json.load(f)
            
        print(f" 模型参数 (从 {self.params_path}):")
        for key, value in model_params.items():
            print(f"   {key}: {value}")
            
        # 2. 加载模型权重
        print(f"\n 加载权重 (从 {self.network_path})")
        checkpoint = torch.load(self.network_path, map_location='cpu')
        
        # 分析checkpoint结构
        if isinstance(checkpoint, OrderedDict):
            state_dict = checkpoint
        elif isinstance(checkpoint, dict):
            # 可能的键
            possible_keys = ['state_dict', 'model_state_dict', 'model', 'network']
            state_dict = None
            
            for key in possible_keys:
                if key in checkpoint:
                    state_dict = checkpoint[key]
                    print(f"   使用键 '{key}' 提取state_dict")
                    break
                    
            if state_dict is None:
                state_dict = checkpoint
        else:
            raise ValueError(f"未知的checkpoint格式: {type(checkpoint)}")
            
        print(f"   加载了 {len(state_dict)} 个层的权重")
        
        # 显示部分层信息
        print(f"\n 权重层预览:")
        for i, (name, param) in enumerate(state_dict.items()):
            if i < 5:
                print(f"   {name}: {list(param.shape)}")
        print(f"   ... (共 {len(state_dict)} 层)")
        
        return model_params, state_dict

# ============================================================================
# Part 3: TabNet模型定义 (匹配预训练结构)
# ============================================================================

class TabNet(nn.Module):
    """
    TabNet模型 - 匹配预训练权重的结构
    """
    
    def __init__(self, input_dim=341, output_dim=8, 
                 n_d=64, n_a=64, n_steps=5, gamma=1.5,
                 cat_idxs=[], cat_dims=[], cat_emb_dim=1,
                 n_independent=2, n_shared=2,
                 epsilon=1e-15, virtual_batch_size=128, momentum=0.02,
                 mask_type="sparsemax"):
        super().__init__()
        
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.n_d = n_d
        self.n_a = n_a
        self.n_steps = n_steps
        self.gamma = gamma
        self.epsilon = epsilon
        self.n_independent = n_independent
        self.n_shared = n_shared
        self.virtual_batch_size = virtual_batch_size
        self.mask_type = mask_type
        
        # 如果没有分类特征，跳过embedding
        if len(cat_idxs) == 0:
            self.embedder = None
            embeddings_size = input_dim
        else:
            # 创建embedding层
            self.embedder = nn.ModuleList()
            for cat_dim in cat_dims:
                emb_dim = min(cat_emb_dim, (cat_dim + 1) // 2)
                self.embedder.append(nn.Embedding(cat_dim, emb_dim))
            embeddings_size = input_dim  # 需要根据实际计算
        
        # Initial batch normalization
        self.initial_bn = nn.BatchNorm1d(embeddings_size, momentum=momentum)
        
        # Shared layers
        if n_shared > 0:
            self.shared = nn.ModuleList()
            for i in range(n_shared):
                if i == 0:
                    self.shared.append(nn.Linear(embeddings_size, 2 * (n_d + n_a), bias=False))
                else:
                    self.shared.append(nn.Linear(n_d + n_a, 2 * (n_d + n_a), bias=False))
        else:
            self.shared = None
            
        # Decision steps
        self.feat_transformers = nn.ModuleList()
        self.att_transformers = nn.ModuleList()
        
        for step in range(n_steps):
            # Feature transformer
            transformer = nn.ModuleList()
            for i in range(n_independent):
                if i == 0:
                    transformer.append(nn.Linear(embeddings_size, 2 * (n_d + n_a), bias=False))
                else:
                    transformer.append(nn.Linear(n_d + n_a, 2 * (n_d + n_a), bias=False))
                    
            self.feat_transformers.append(transformer)
            
            # Attention transformer
            self.att_transformers.append(
                AttentiveTransformer(n_a, embeddings_size, virtual_batch_size, 
                                   momentum, mask_type)
            )
        
        # Final mapping
        self.final_mapping = nn.Linear(n_d, output_dim, bias=False)
        
    def forward(self, x):
        # Embedding (如果需要)
        if self.embedder is not None:
            # 处理分类特征
            pass  # 这里简化，因为您的数据都是连续特征
            
        # Initial normalization
        x = self.initial_bn(x)
        batch_size = x.shape[0]
        
        # Initialize prior
        prior = torch.ones((batch_size, self.input_dim)).to(x.device)
        
        # Initialize attention and output  
        M_loss = 0
        att = self.initial_splitter(x)
        
        # Accumulate decision outputs
        out = torch.zeros((batch_size, self.n_d)).to(x.device)
        
        # Decision steps
        for step in range(self.n_steps):
            # Attention
            M = self.att_transformers[step](prior, att)
            M_loss += torch.mean(
                torch.sum(torch.mul(M, torch.log(M + self.epsilon)), dim=1)
            )
            
            # Update prior
            prior = torch.mul(self.gamma - M, prior)
            
            # Apply mask
            masked_x = torch.mul(M, x)
            
            # Transform features
            out_step = self.feat_transformers[step][0](masked_x)
            for i in range(1, self.n_independent):
                out_step = self.feat_transformers[step][i](out_step[:, :self.n_d + self.n_a])
            
            # GLU activation
            out_step = F.glu(out_step, dim=1)
            
            # Accumulate
            out += out_step[:, :self.n_d]
            
            # Update attention features
            att = out_step[:, self.n_d:]
            
        # Final mapping
        out = self.final_mapping(out)
        
        return out, M_loss
    
    def initial_splitter(self, x):
        """Initial GLU block"""
        if self.n_shared > 0:
            x = self.shared[0](x)
        else:
            x = self.feat_transformers[0][0](x)
            
        # GLU split
        return F.glu(x, dim=1)[:, self.n_d:]

class AttentiveTransformer(nn.Module):
    """注意力转换器"""
    
    def __init__(self, n_a, input_dim, virtual_batch_size, momentum, mask_type):
        super().__init__()
        self.fc = nn.Linear(n_a, input_dim, bias=False)
        self.bn = nn.BatchNorm1d(input_dim, momentum=momentum)
        self.mask_type = mask_type
        
    def forward(self, prior, att):
        a = self.fc(att)
        a = self.bn(a)
        a = torch.mul(prior, a)
        
        if self.mask_type == "sparsemax":
            mask = self.sparsemax(a)
        else:
            mask = F.softmax(a, dim=-1)
            
        return mask
    
    def sparsemax(self, input):
        """Sparsemax activation"""
        dim = 1
        input = input - torch.max(input, dim=dim, keepdim=True)[0]
        
        zs = torch.sort(input, dim=dim, descending=True)[0]
        range_vals = torch.arange(1, input.size(dim) + 1).float().to(input.device)
        range_vals = range_vals.unsqueeze(0).expand_as(zs)
        
        bound = 1 + range_vals * zs
        cumsum_zs = torch.cumsum(zs, dim)
        is_gt = bound > cumsum_zs
        k = torch.max(is_gt * range_vals, dim, keepdim=True)[0]
        
        zs_sparse = is_gt * zs
        taus = (torch.sum(zs_sparse, dim, keepdim=True) - 1) / k
        taus = taus.expand_as(input)
        
        output = torch.max(torch.zeros_like(input), input - taus)
        return output

# ============================================================================
# Part 4: Lightning模块 - 核心迁移学习实现
# ============================================================================

class TabNetTransferLearning(pl.LightningModule):
    """
     PyTorch Lightning模块
    - 支持逐步解冻
    - 监控3个数据集的Loss和Macro F1
    - 使用预训练权重
    """
    
    def __init__(self, 
                 pretrained_dir: str = 'tabnet_model_test0',
                 num_classes: int = 102,
                 learning_rate: float = 1e-3,
                 weight_decay: float = 1e-5,
                 freeze_stage: int = 0):
        super().__init__()
        
        # 保存超参数
        self.save_hyperparameters()
        
        # 加载预训练模型
        loader = PretrainedTabNetLoader(pretrained_dir)
        model_params, pretrained_weights = loader.load()
        
        # 创建TabNet backbone
        self.tabnet = TabNet(
            input_dim=341,  # 固定输入维度
            output_dim=model_params.get('n_d', 64),  # 预训练输出维度
            n_d=model_params.get('n_d', 64),
            n_a=model_params.get('n_a', 64),
            n_steps=model_params.get('n_steps', 5),
            gamma=model_params.get('gamma', 1.5),
            n_independent=model_params.get('n_independent', 2),
            n_shared=model_params.get('n_shared', 2),
            epsilon=model_params.get('epsilon', 1e-15),
            virtual_batch_size=model_params.get('virtual_batch_size', 128),
            momentum=model_params.get('momentum', 0.02),
            mask_type=model_params.get('mask_type', 'sparsemax')
        )
        
        #  加载预训练权重
        self._load_pretrained_weights(pretrained_weights)
        
        # 添加任务特定的分类头
        tabnet_output_dim = model_params.get('n_d', 64)
        self.classifier = nn.Sequential(
            nn.Linear(tabnet_output_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            
            nn.Linear(256, num_classes)
        )
        
        #  设置初始冻结状态
        self.freeze_stage = freeze_stage
        self.configure_freezing()
        
        # 用于记录每个epoch的指标
        self.epoch_metrics = {
            'train': {'loss': [], 'f1': []},
            'val': {'loss': [], 'f1': []},
            'test': {'loss': [], 'f1': []}
        }
        
    def _load_pretrained_weights(self, pretrained_weights):
        """ 加载预训练权重"""
        # 获取当前模型的state dict
        model_dict = self.tabnet.state_dict()
        
        # 匹配并加载权重
        matched_keys = []
        unmatched_keys = []
        
        for key, value in pretrained_weights.items():
            if key in model_dict:
                if value.shape == model_dict[key].shape:
                    model_dict[key] = value
                    matched_keys.append(key)
                else:
                    print(f" 形状不匹配: {key}")
                    unmatched_keys.append(key)
            else:
                # 尝试处理命名差异
                cleaned_key = key.replace('module.', '').replace('model.', '')
                if cleaned_key in model_dict and value.shape == model_dict[cleaned_key].shape:
                    model_dict[cleaned_key] = value
                    matched_keys.append(cleaned_key)
                else:
                    unmatched_keys.append(key)
        
        # 更新模型权重
        self.tabnet.load_state_dict(model_dict)
        
        print(f"\n 预训练权重加载完成:")
        print(f"   成功匹配: {len(matched_keys)} 层")
        print(f"   未匹配: {len(unmatched_keys)} 层")
        
    def configure_freezing(self):
        """ 配置冻结策略 - 逐步解冻的核心"""
        print(f"\n🔧 配置冻结策略 - Stage {self.freeze_stage}")
        
        if self.freeze_stage == 0:
            # Stage 0: 冻结整个TabNet
            for param in self.tabnet.parameters():
                param.requires_grad = False
            print("❄️ Stage 0: 冻结整个TabNet backbone")
            
        elif self.freeze_stage == 1:
            # Stage 1: 解冻最后2个decision steps
            for param in self.tabnet.parameters():
                param.requires_grad = False
                
            # 解冻最后2个steps
            n_steps = len(self.tabnet.feat_transformers)
            for i in range(max(0, n_steps-2), n_steps):
                for param in self.tabnet.feat_transformers[i].parameters():
                    param.requires_grad = True
                for param in self.tabnet.att_transformers[i].parameters():
                    param.requires_grad = True
                    
            # 解冻final mapping
            for param in self.tabnet.final_mapping.parameters():
                param.requires_grad = True
                
            print(f" Stage 1: 解冻最后2个decision steps (共{n_steps}个)")
            
        elif self.freeze_stage == 2:
            # Stage 2: 解冻所有decision steps，保持shared layers冻结
            for param in self.tabnet.parameters():
                param.requires_grad = False
                
            # 解冻所有decision steps
            for transformer in self.tabnet.feat_transformers:
                for param in transformer.parameters():
                    param.requires_grad = True
                    
            for transformer in self.tabnet.att_transformers:
                for param in transformer.parameters():
                    param.requires_grad = True
                    
            # 解冻final mapping
            for param in self.tabnet.final_mapping.parameters():
                param.requires_grad = True
                
            # BN层保持冻结
            print(" Stage 2: 解冻所有decision steps")
            
        else:  # Stage 3
            # Stage 3: 全部解冻
            for param in self.tabnet.parameters():
                param.requires_grad = True
            print(" Stage 3: 全部解冻 - 微调整个网络")
        
        # 确保分类头始终可训练
        for param in self.classifier.parameters():
            param.requires_grad = True
            
        # 统计可训练参数
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        frozen_params = total_params - trainable_params
        
        print(f"\n 参数统计:")
        print(f"   总参数: {total_params:,}")
        print(f"   可训练: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
        print(f"   冻结: {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")
        
    def forward(self, x):
        # TabNet编码
        encoded, M_loss = self.tabnet(x)
        
        # 分类
        logits = self.classifier(encoded)
        
        return logits, M_loss
    
    def _compute_loss_and_metrics(self, batch, stage='train'):
        """ 计算损失和Macro F1"""
        x, y = batch
        
        # 处理one-hot标签
        if y.dim() > 1 and y.shape[1] > 1:
            y_true = torch.argmax(y, dim=1)
        else:
            y_true = y.long()
        
        # 前向传播
        logits, M_loss = self(x)
        
        # 计算损失
        ce_loss = F.cross_entropy(logits, y_true)
        sparsity_loss = 1e-3 * M_loss  # TabNet稀疏性正则化
        total_loss = ce_loss + sparsity_loss
        
        #  计算Macro F1 - 核心评估指标
        with torch.no_grad():
            y_pred = torch.argmax(logits, dim=1)
            macro_f1 = f1_score(
                y_true.cpu().numpy(), 
                y_pred.cpu().numpy(),
                average='macro',
                zero_division=0
            )
        
        return total_loss, macro_f1
    
    def training_step(self, batch, batch_idx):
        """ 训练步骤"""
        loss, macro_f1 = self._compute_loss_and_metrics(batch, 'train')
        
        # 记录指标
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('train_f1', macro_f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        """ 验证步骤"""
        loss, macro_f1 = self._compute_loss_and_metrics(batch, 'val')
        
        # 记录指标
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('val_f1', macro_f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return {'val_loss': loss, 'val_f1': macro_f1}
    
    def test_step(self, batch, batch_idx):
        """ 测试步骤"""
        loss, macro_f1 = self._compute_loss_and_metrics(batch, 'test')
        
        # 记录指标
        self.log('test_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log('test_f1', macro_f1, on_step=False, on_epoch=True, prog_bar=True)
        
        return {'test_loss': loss, 'test_f1': macro_f1}
    
    def configure_optimizers(self):
        """ 配置优化器 - 分层学习率"""
        # 分组参数
        param_groups = []
        
        if self.freeze_stage == 0:
            # Stage 0: 只优化分类头
            param_groups.append({
                'params': self.classifier.parameters(),
                'lr': self.hparams.learning_rate,
                'name': 'classifier'
            })
        else:
            # 其他阶段: TabNet使用更小的学习率
            lr_scale = 0.1 ** (3 - self.freeze_stage)  # Stage 1: 0.01, Stage 2: 0.1, Stage 3: 1.0
            
            # TabNet参数 (可训练的)
            tabnet_params = [p for p in self.tabnet.parameters() if p.requires_grad]
            if tabnet_params:
                param_groups.append({
                    'params': tabnet_params,
                    'lr': self.hparams.learning_rate * lr_scale,
                    'name': 'tabnet'
                })
            
            # 分类头参数
            param_groups.append({
                'params': self.classifier.parameters(),
                'lr': self.hparams.learning_rate,
                'name': 'classifier'
            })
        
        # 创建优化器
        optimizer = torch.optim.AdamW(
            param_groups,
            weight_decay=self.hparams.weight_decay
        )
        
        # 学习率调度器
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=10,
            eta_min=1e-6
        )
        
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'frequency': 1,
                'monitor': 'val_f1'
            }
        }
    
    def on_train_epoch_end(self):
        """记录训练epoch结束时的指标"""
        metrics = self.trainer.logged_metrics
        self.epoch_metrics['train']['loss'].append(metrics.get('train_loss', torch.tensor(0)).item())
        self.epoch_metrics['train']['f1'].append(metrics.get('train_f1', torch.tensor(0)).item())
        
    def on_validation_epoch_end(self):
        """记录验证epoch结束时的指标"""
        metrics = self.trainer.logged_metrics
        self.epoch_metrics['val']['loss'].append(metrics.get('val_loss', torch.tensor(0)).item())
        self.epoch_metrics['val']['f1'].append(metrics.get('val_f1', torch.tensor(0)).item())

# ============================================================================
# Part 5: 逐步解冻回调
# ============================================================================

class ProgressiveUnfreezing(Callback):
    """
     逐步解冻回调 - 基于验证集Macro F1自动切换阶段
    """
    
    def __init__(self, patience: int = 3, min_delta: float = 0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.wait = 0
        self.best_f1 = -float('inf')
        self.current_stage = 0
        
    def on_validation_epoch_end(self, trainer, pl_module):
        # 获取当前验证F1
        current_f1 = trainer.callback_metrics.get('val_f1', torch.tensor(0)).item()
        
        # 检查是否有改善
        if current_f1 > self.best_f1 + self.min_delta:
            self.best_f1 = current_f1
            self.wait = 0
        else:
            self.wait += 1
            
        print(f"\n Epoch {trainer.current_epoch}: "
              f"Val F1 = {current_f1:.4f} (best = {self.best_f1:.4f}), "
              f"wait = {self.wait}/{self.patience}")
        
        #  判断是否进入下一阶段
        if self.wait >= self.patience and self.current_stage < 3:
            self.current_stage += 1
            self.wait = 0
            
            print(f"\n 验证F1停滞 {self.patience} epochs，进入 Stage {self.current_stage}")
            
            # 更新模型的解冻策略
            pl_module.freeze_stage = self.current_stage
            pl_module.configure_freezing()
            
            # 重新配置优化器
            trainer.strategy.setup_optimizers(trainer)
            
            # 重置最佳F1（给新阶段一个机会）
            self.best_f1 = current_f1

class TestDuringValidation(Callback):
    """
     在验证时同时进行测试，监控3个数据集的指标
    """
    
    def __init__(self):
        self.test_results = []
        
    def on_validation_epoch_end(self, trainer, pl_module):
        # 在验证结束后立即测试
        if hasattr(trainer, 'test_dataloaders') and trainer.test_dataloaders:
            print(f"   运行测试集评估...")
            test_results = trainer.test(pl_module, trainer.test_dataloaders, verbose=False)
            
            if test_results and len(test_results) > 0:
                test_loss = test_results[0].get('test_loss', 0)
                test_f1 = test_results[0].get('test_f1', 0)
                
                pl_module.epoch_metrics['test']['loss'].append(test_loss)
                pl_module.epoch_metrics['test']['f1'].append(test_f1)
                
                # 打印所有3个数据集的结果
                train_f1 = trainer.callback_metrics.get('train_f1', torch.tensor(0)).item()
                val_f1 = trainer.callback_metrics.get('val_f1', torch.tensor(0)).item()
                
                print(f"\n Epoch {trainer.current_epoch} - 所有数据集Macro F1:")
                print(f"   Train: {train_f1:.4f}")
                print(f"   Val:   {val_f1:.4f}")
                print(f"   Test:  {test_f1:.4f}")

# ============================================================================
# Part 6: 主训练函数
# ============================================================================

def train_tabnet_transfer():
    """
    主训练函数
     完整实现所有要求
    """
    
    # 配置
    config = {
        'data_path': '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat',
        'pretrained_dir': 'tabnet_model_test0',
        'batch_size': 256,
        'num_epochs': 40,
        'learning_rate': 1e-3,
        'weight_decay': 1e-5,
        'patience': 3,
        'num_workers': 4,
    }
    
    print("="*80)
    print("🧠 TabNet迁移学习训练")
    print("="*80)
    print("\n📋 训练配置:")
    for key, value in config.items():
        print(f"   {key}: {value}")
    
    # 1. 准备数据 ( 包含标准化)
    data_module = BrainVoxelDataModule(
        data_path=config['data_path'],
        batch_size=config['batch_size'],
        num_workers=config['num_workers']
    )
    data_module.setup()
    
    # 2. 创建模型 ( 加载预训练权重)
    model = TabNetTransferLearning(
        pretrained_dir=config['pretrained_dir'],
        num_classes=102,
        learning_rate=config['learning_rate'],
        weight_decay=config['weight_decay'],
        freeze_stage=0  # 从Stage 0开始
    )
    
    # 3. 设置回调 ( 逐步解冻 + 监控3个数据集)
    callbacks = [
        ProgressiveUnfreezing(patience=config['patience']),
        TestDuringValidation(),
        ModelCheckpoint(
            monitor='val_f1',
            mode='max',
            save_top_k=3,
            filename='tabnet-{epoch:02d}-{val_f1:.4f}',
            verbose=True
        ),
        EarlyStopping(
            monitor='val_f1',
            patience=10,
            mode='max',
            verbose=True
        ),
        LearningRateMonitor(logging_interval='epoch')
    ]
    
    # 4. 创建训练器
    trainer = pl.Trainer(
        max_epochs=config['num_epochs'],
        callbacks=callbacks,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision=16,  # 混合精度训练
        gradient_clip_val=1.0,
        deterministic=True,
        log_every_n_steps=50,
        enable_progress_bar=True,
        enable_checkpointing=True,
    )
    
    # 5. 设置测试数据加载器（用于验证时测试）
    trainer.test_dataloaders = data_module.test_dataloader()
    
    # 6. 训练 ( 基于Lightning的训练循环)
    print("\n 开始训练...")
    print("="*80)
    
    trainer.fit(
        model,
        train_dataloaders=data_module.train_dataloader(),
        val_dataloaders=data_module.val_dataloader()
    )
    
    # 7. 最终测试
    print("\n🔍 最终测试...")
    test_results = trainer.test(model, data_module.test_dataloader())
    
    # 8. 返回结果
    return model, model.epoch_metrics, test_results

# ============================================================================
# Part 7: 结果可视化
# ============================================================================

def plot_training_results(metrics_history):
    """
     绘制训练结果 - 显示3个数据集的Loss和Macro F1
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    epochs = range(len(metrics_history['train']['loss']))
    
    # 配色
    colors = {
        'train': '#1f77b4',  # 蓝色
        'val': '#ff7f0e',    # 橙色
        'test': '#2ca02c'    # 绿色
    }
    
    # 1. 训练Loss
    ax = axes[0, 0]
    ax.plot(epochs, metrics_history['train']['loss'], 
            color=colors['train'], linewidth=2, label='Train')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training Loss', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # 2. 验证Loss
    ax = axes[0, 1]
    ax.plot(epochs, metrics_history['val']['loss'], 
            color=colors['val'], linewidth=2, label='Val')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Validation Loss', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # 3. 测试Loss
    ax = axes[0, 2]
    if metrics_history['test']['loss']:
        test_epochs = range(len(metrics_history['test']['loss']))
        ax.plot(test_epochs, metrics_history['test']['loss'], 
                color=colors['test'], linewidth=2, label='Test')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Test Loss', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # 4. 训练F1
    ax = axes[1, 0]
    ax.plot(epochs, metrics_history['train']['f1'], 
            color=colors['train'], linewidth=2, label='Train')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Macro F1')
    ax.set_title('Training Macro F1', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_ylim([0, 1])
    
    # 5. 验证F1
    ax = axes[1, 1]
    ax.plot(epochs, metrics_history['val']['f1'], 
            color=colors['val'], linewidth=2, label='Val')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Macro F1')
    ax.set_title('Validation Macro F1', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_ylim([0, 1])
    
    # 6. 测试F1
    ax = axes[1, 2]
    if metrics_history['test']['f1']:
        test_epochs = range(len(metrics_history['test']['f1']))
        ax.plot(test_epochs, metrics_history['test']['f1'], 
                color=colors['test'], linewidth=2, label='Test')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Macro F1')
    ax.set_title('Test Macro F1', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_ylim([0, 1])
    
    plt.suptitle('TabNet Transfer Learning Results - All Datasets', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('tabnet_transfer_results_all_datasets.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # 打印最终结果
    print("\n" + "="*80)
    print(" 最终结果总结")
    print("="*80)
    
    print("\n 最终Macro F1:")
    print(f"   Train: {metrics_history['train']['f1'][-1]:.4f}")
    print(f"   Val:   {metrics_history['val']['f1'][-1]:.4f}")
    if metrics_history['test']['f1']:
        print(f"   Test:  {metrics_history['test']['f1'][-1]:.4f}")
    
    print("\n 最终Loss:")
    print(f"   Train: {metrics_history['train']['loss'][-1]:.4f}")
    print(f"   Val:   {metrics_history['val']['loss'][-1]:.4f}")
    if metrics_history['test']['loss']:
        print(f"   Test:  {metrics_history['test']['loss'][-1]:.4f}")
    
    print("\n 性能提升:")
    print(f"   Val F1提升: {metrics_history['val']['f1'][-1] - metrics_history['val']['f1'][0]:.4f}")
    if metrics_history['test']['f1']:
        print(f"   Test F1提升: {metrics_history['test']['f1'][-1] - metrics_history['test']['f1'][0]:.4f}")

# ============================================================================
# 主执行入口
# ============================================================================

if __name__ == "__main__":
    # 训练模型
    model, metrics_history, test_results = train_tabnet_transfer()
    
    # 绘制结果
    plot_training_results(metrics_history)
    
    print("\n 训练完成！")